# Parsing GeoGebra Construction Protocolto Polars DataFrame

In [ ]:
# this magic for develop only
%load_ext autoreload
%autoreload 2

In [1]:
from ggblab import GeoGebra

In [2]:
# initialize base class, not open GeoGebra Widget
ggb = GeoGebra()

Using local cached file: xsd/common.xsd


In [3]:
from ggblab_extra.construction_io import ConstructionIO

In [4]:
c = ggb.file.load('2025_13_01.ggb')

In [5]:
# df1 from a file df2 from a applet
df1 = await ConstructionIO.initialize_dataframe(ggb, file='2025_13_01.ggb')

In [6]:
df1

Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary
str,str,str,str,str,i64,bool,bool,bool
"""C""","""point""",null,null,null,9,true,true,false
"""A""","""point""",null,null,null,9,true,true,false
"""poly1""","""polygon""","""Polygon(C, A, 4)""",null,null,3,false,false,true
"""f""","""segment""","""Segment(C, A, poly1)""",null,null,2,false,false,true
"""g""","""segment""","""Segment(A, E, poly1)""",null,null,2,false,false,true
"""h""","""segment""","""Segment(E, D, poly1)""",null,null,2,false,false,true
"""i""","""segment""","""Segment(D, C, poly1)""",null,null,2,false,false,true
"""E""","""point""","""Polygon(C, A, 4)""",null,null,2,true,false,true
"""D""","""point""","""Polygon(C, A, 4)""",null,null,2,true,false,true


In [7]:
await ggb.init()

In [8]:
r = await ggb.function("setBase64", [ggb.construction.base64_buffer.decode('utf-8')])

In [9]:
# df1 from a file df2 from a applet
df2 = await ConstructionIO.initialize_dataframe(ggb, use_applet=True)

In [10]:
ggb.file.source_file

'2025_13_01.ggb'

In [11]:
ConstructionIO.save_dataframe(df2, ggb=ggb, fmt='json')

'2025_13_01_2.json'

In [12]:
df2

Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary
str,str,str,str,str,i64,bool,bool,bool
"""C""","""point""",null,"""C = (0, 0)""",null,9,true,true,false
"""A""","""point""",null,"""A = (2.2, 0)""",null,9,true,true,false
"""poly1""","""polygon""","""Polygon(C, A, 4)""","""poly1 = 4.6""",null,3,false,false,true
"""f""","""segment""","""Segment(C, A, poly1)""","""f = 2.2""",null,2,false,false,true
"""g""","""segment""","""Segment(A, E, poly1)""","""g = 2.2""",null,2,false,false,true
"""E""","""point""","""Polygon(C, A, 4)""","""E = (2.1, 2.2)""",null,2,true,false,true
"""D""","""point""","""Polygon(C, A, 4)""","""D = (0, 2.2)""",null,2,true,false,true
"""h""","""segment""","""Segment(E, D, poly1)""","""h = 2.2""",null,2,false,false,true
"""i""","""segment""","""Segment(D, C, poly1)""","""i = 2.2""",null,2,false,false,true


In [13]:
set(df1["Type"].unique()) - set(df2["Type"].unique())

{'conic'}

In [14]:
set(df2["Type"].unique()) - set(df1["Type"].unique())

{'circle', 'quadrilateral', 'triangle'}

In [15]:
# df1 and df2 have different order...
# df1["Command"] == df2["Command"]
mask = df1['Command'].eq_missing(df2['Command']).not_()
# df1.filter(mask)
df2.filter(mask)

Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary
str,str,str,str,str,i64,bool,bool,bool
"""E""","""point""","""Polygon(C, A, 4)""","""E = (2.1, 2.2)""",null,2,true,false,true
"""D""","""point""","""Polygon(C, A, 4)""","""D = (0, 2.2)""",null,2,true,false,true
"""h""","""segment""","""Segment(E, D, poly1)""","""h = 2.2""",null,2,false,false,true
"""i""","""segment""","""Segment(D, C, poly1)""","""i = 2.2""",null,2,false,false,true
"""G""","""point""","""Polygon(A, C, 4)""","""G = (0, -2.2)""",null,4,true,false,true
"""H""","""point""","""Polygon(A, C, 4)""","""H = (2.2, -2.1)""",null,4,true,false,true
"""a_{3}""","""segment""","""Segment(G, H, poly2)""","""a_{3} = 2.2""",null,4,false,false,true
"""b_1""","""segment""","""Segment(H, A, poly2)""","""b_1 = 2.2""",null,4,false,false,true
"""proj_{u}w""","""numeric""","""(w u) / (u u)""","""proj_{u}w = 1.2""",null,8,false,false,false


## IR files

In [ ]:
import os
os.path.splitext(ggb.file.source_file)[0]+'.json'

In [ ]:
df1.write_json(os.path.splitext(ggb.file.source_file)[0]+'.json')

In [ ]:
import xml.etree.ElementTree as ET
import xmltodict

In [ ]:
root = ET.Element(c.geogebra_xml)
tree = ET.ElementTree(root)
tree.write(os.path.splitext(ggb.file.source_file)[0]+'.xml', encoding='utf-8', xml_declaration=True)

## handle no root returens from the applet

In [16]:
r = await ggb.function("getXML", ["text1"])

In [17]:
print(r)

<expression label="text1" exp="&quot;1.  Thales&apos;s  theorem: right  triangle  inscribed  in  a  circle&quot;"/>
<element type="text" label="text1">
	<show object="true" label="false" ev="40"/>
	<objColor r="0" g="0" b="0" alpha="0"/>
	<layer val="9"/>
	<labelMode val="0"/>
	<isLaTeX val="true"/>
	<font serif="false" sizeM="1" size="0" style="0"/>
	<absoluteScreenLocation x="50" y="50"/>
</element>



In [18]:
import xml.etree.ElementTree as ET
from itertools import chain

try:
    o3 = ggb.file.ggb_schema.decode(r)
except ET.ParseError:
    vr = ET.fromstringlist(chain(['<construction>'], r, ['</construction>']))
    o3 = ggb.file.ggb_schema.decode(ET.tostring(vr).decode('utf-8'))
o3

{'expression': [{'@label': 'text1',
   '@exp': '"1. \xa0Thales\'s \xa0theorem: right \xa0triangle \xa0inscribed \xa0in \xa0a \xa0circle"'}],
 'element': [{'@type': 'text',
   '@label': 'text1',
   'show': [{'@object': True, '@label': False, '@ev': 40}],
   'objColor': [{'@r': 0, '@g': 0, '@b': 0, '@alpha': 0.0}],
   'layer': [{'@val': 9}],
   'labelMode': [{'@val': 0}],
   'isLaTeX': [{'@val': True}],
   'font': [{'@serif': False, '@sizeM': 1.0, '@size': 0, '@style': 0}],
   'absoluteScreenLocation': [{'@x': 50.0, '@y': 50.0}]}]}

In [19]:
o3['element'][0].get('show')

[{'@object': True, '@label': False, '@ev': 40}]